# Short-term stock market prediction with Time Series Transformer Network	

In [14]:
import torch
import torch.nn as nn
import numpy as np

import pandas as pd

In [15]:
data_all = pd.read_csv("datasets/store-sales-dataset/train.csv")
#data_all.replace('?', np.nan, inplace=True)
data_all

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.000,0
1,1,2013-01-01,1,BABY CARE,0.000,0
2,2,2013-01-01,1,BEAUTY,0.000,0
3,3,2013-01-01,1,BEVERAGES,0.000,0
4,4,2013-01-01,1,BOOKS,0.000,0
...,...,...,...,...,...,...
3000883,3000883,2017-08-15,9,POULTRY,438.133,0
3000884,3000884,2017-08-15,9,PREPARED FOODS,154.553,1
3000885,3000885,2017-08-15,9,PRODUCE,2419.729,148
3000886,3000886,2017-08-15,9,SCHOOL AND OFFICE SUPPLIES,121.000,8


In [18]:
categorical_covariates = ['date']#'time_idx',,'week_day','month_day','month','year','holiday']

categorical_covariates_num_embeddings = []
for col in categorical_covariates:
    data_all[col] = data_all[col].astype('category').cat.codes
    categorical_covariates_num_embeddings.append(data_all[col].nunique())

categorical_static = ['store_nbr']#,'family']#'city','state','type','cluster','family_int'] #family giving errors

categorical_static_num_embeddings = []
for col in categorical_static:
    data_all[col] = data_all[col].astype('category').cat.codes
    categorical_static_num_embeddings.append(data_all[col].nunique())

numeric_covariates = ['sales','onpromotion']#'dcoilwtico','dcoilwtico_future','onpromotion','onpromotion_future','store_sales','transactions','family_sales']

target_idx = np.where(np.array(numeric_covariates)=='sales')[0][0]

In [ ]:
def dataframe_to_tensor(series,numeric_covariates,categorical_covariates,categorical_static,target_idx):

    numeric_cov_arr = np.array(series[numeric_covariates].values.tolist())
    category_cov_arr = np.array(series[categorical_covariates].values.tolist())
    static_cov_arr = np.array(series[categorical_static].values.tolist())

    x_numeric = torch.tensor(numeric_cov_arr,dtype=torch.float32).transpose(2,1)
    x_numeric = torch.log(x_numeric+1e-5)
    x_category = torch.tensor(category_cov_arr,dtype=torch.long).transpose(2,1)
    x_static = torch.tensor(static_cov_arr,dtype=torch.long)
    y = torch.tensor(numeric_cov_arr[:,target_idx,:],dtype=torch.float32)

    return x_numeric, x_category, x_static, y


window_size = 16
forecast_length = 16
num_val = 2

data_all['date'] = pd.to_datetime(data_all['date']) #convert int to datetime
val_max_date = '2017-08-15'
train_max_date = str((pd.to_datetime(val_max_date) - pd.Timedelta(days=window_size*num_val+forecast_length)).date())

train_final = data_all[data_all['date']<=train_max_date]
val_final = data_all[(data_all['date']>train_max_date)&(data_all['date']<=val_max_date)]

train_series = train_final.groupby(categorical_static).agg(list).reset_index()
val_series = val_final.groupby(categorical_static).agg(list).reset_index()

#try changing date to ints
train_series
data_all['date'] = pd.to_datetime(data_all['date']) #convert int to datetime

x_numeric_train_tensor, x_category_train_tensor, x_static_train_tensor, y_train_tensor = dataframe_to_tensor(train_series,numeric_covariates,categorical_covariates,categorical_static,target_idx)

x_numeric_val_tensor, x_category_val_tensor, x_static_val_tensor, y_val_tensor = dataframe_to_tensor(val_series,numeric_covariates,categorical_covariates,categorical_static,target_idx)

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [3]:
def divide_shuffle(df,div_num):
    space = df.shape[0]//div_num
    division = np.arange(0,df.shape[0],space)
    return pd.concat([df.iloc[division[i]:division[i]+space,:].sample(frac=1) for i in range(len(division))])

def create_time_blocks(time_length,window_size,forecast_length):
    start_idx = np.random.randint(0,window_size-1)
    end_idx = time_length-window_size-forecast_length-1
    time_indices = np.arange(start_idx,end_idx+1,window_size)[:-1]
    time_indices = np.append(time_indices,end_idx)
    return time_indices

def data_loader(x_numeric_tensor, x_category_tensor, x_static_tensor, y_tensor, batch_size, time_shuffle):

    num_series = x_numeric_tensor.shape[0]
    time_length = x_numeric_tensor.shape[1]
    index_pd = pd.DataFrame({'serie_idx':range(num_series)})
    index_pd['time_idx'] = [create_time_blocks(time_length,window_size,forecast_length) for n in range(index_pd.shape[0])]
    if time_shuffle:
        index_pd = index_pd.explode('time_idx')
        index_pd = index_pd.sample(frac=1)
    else:
        index_pd = index_pd.explode('time_idx').sort_values('time_idx')
        index_pd = divide_shuffle(index_pd,5)
    indices = np.array(index_pd).astype(int)

    for batch_idx in np.arange(0,indices.shape[0],batch_size):

        cur_indices = indices[batch_idx:batch_idx+batch_size,:]

        x_numeric = torch.stack([x_numeric_tensor[n[0],n[1]:n[1]+window_size,:] for n in cur_indices])
        x_category = torch.stack([x_category_tensor[n[0],n[1]:n[1]+window_size,:] for n in cur_indices])
        x_static = torch.stack([x_static_tensor[n[0],:] for n in cur_indices])
        y = torch.stack([y_tensor[n[0],n[1]+window_size:n[1]+window_size+forecast_length] for n in cur_indices])

        yield x_numeric.to(device), x_category.to(device), x_static.to(device), y.to(device)

def val_loader(x_numeric_tensor, x_category_tensor, x_static_tensor, y_tensor, batch_size, num_val):

    num_time_series = x_numeric_tensor.shape[0]

    for i in range(num_val):

      for batch_idx in np.arange(0,num_time_series,batch_size):

          x_numeric = x_numeric_tensor[batch_idx:batch_idx+batch_size,window_size*i:window_size*(i+1),:]
          x_category = x_category_tensor[batch_idx:batch_idx+batch_size,window_size*i:window_size*(i+1),:]
          x_static = x_static_tensor[batch_idx:batch_idx+batch_size]
          y_val = y_tensor[batch_idx:batch_idx+batch_size,window_size*(i+1):window_size*(i+1)+forecast_length]

          yield x_numeric.to(device), x_category.to(device), x_static.to(device), y_val.to(device)

In [10]:
class transformer_block(nn.Module):

    def __init__(self,embed_size,num_heads):
        super(transformer_block, self).__init__()

        self.attention = nn.MultiheadAttention(embed_size, num_heads, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(embed_size, 4 * embed_size),
                                 nn.LeakyReLU(),
                                 nn.Linear(4 * embed_size, embed_size))
        self.dropout = nn.Dropout(drop_prob)
        self.ln1 = nn.LayerNorm(embed_size, eps=1e-6)
        self.ln2 = nn.LayerNorm(embed_size, eps=1e-6)

    def forward(self, x):

        attn_out, _ = self.attention(x, x, x, need_weights=False)
        x = x + self.dropout(attn_out)
        x = self.ln1(x)

        fc_out = self.fc(x)
        x = x + self.dropout(fc_out)
        x = self.ln2(x)

        return x

class transformer_forecaster(nn.Module):

    def __init__(self,embed_size,num_heads,num_blocks):
        super(transformer_forecaster, self).__init__()

        num_len = len(numeric_covariates)
        self.embedding_cov = nn.ModuleList([nn.Embedding(n,embed_size-num_len) for n in categorical_covariates_num_embeddings])
        self.embedding_static = nn.ModuleList([nn.Embedding(n,embed_size-num_len) for n in categorical_static_num_embeddings])

        self.blocks = nn.ModuleList([transformer_block(embed_size,num_heads) for n in range(num_blocks)])

        self.forecast_head = nn.Sequential(nn.Linear(embed_size, embed_size*2),
                                           nn.LeakyReLU(),
                                           nn.Dropout(drop_prob),
                                           nn.Linear(embed_size*2, embed_size*4),
                                           nn.LeakyReLU(),
                                           nn.Linear(embed_size*4, forecast_length),
                                           nn.ReLU())

    def forward(self, x_numeric, x_category, x_static):

        tmp_list = []
        for i,embed_layer in enumerate(self.embedding_static):
            tmp_list.append(embed_layer(x_static[:,i]))
        categroical_static_embeddings = torch.stack(tmp_list).mean(dim=0).unsqueeze(1)

        tmp_list = []
        for i,embed_layer in enumerate(self.embedding_cov):
            tmp_list.append(embed_layer(x_category[:,:,i]))
        categroical_covariates_embeddings = torch.stack(tmp_list).mean(dim=0)
        T = categroical_covariates_embeddings.shape[1]

        embed_out = (categroical_covariates_embeddings + categroical_static_embeddings.repeat(1,T,1))/2
        x = torch.concat((x_numeric,embed_out),dim=-1)

        for block in self.blocks:
            x = block(x)

        x = x.mean(dim=1)
        x = self.forecast_head(x)

        return x

In [11]:
class RMSLELoss(nn.Module):

    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, pred, actual):
        return torch.sqrt(self.mse(torch.log(pred + 1), torch.log(actual + 1)))

In [12]:
num_epoch = 1000
min_val_loss = 999

num_blocks = 1
embed_size = 500
num_heads = 50
batch_size = 128
learning_rate = 3e-4
time_shuffle = False
drop_prob = 0.1

model = transformer_forecaster(embed_size,num_heads,num_blocks).to(device)
criterion = RMSLELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

NameError: name 'numeric_covariates' is not defined

In [13]:
for epoch in range(num_epoch):

    batch_loader = data_loader(x_numeric_train_tensor, x_category_train_tensor, x_static_train_tensor, y_train_tensor, batch_size, time_shuffle)
    train_loss = 0
    counter = 0

    model.train()
    for x_numeric, x_category, x_static, y in batch_loader:

        optimizer.zero_grad()
        preds = model(x_numeric, x_category, x_static)
        loss = criterion(preds, y)
        train_loss += loss.item()
        counter += 1
        loss.backward()
        optimizer.step()

    train_loss = train_loss/counter
    print(f'Epoch {epoch} training loss: {train_loss}')

    model.eval()
    val_batches = val_loader(x_numeric_val_tensor, x_category_val_tensor, x_static_val_tensor, y_val_tensor, batch_size, num_val)
    val_loss = 0
    counter = 0
    for x_numeric_val, x_category_val, x_static_val, y_val in val_batches:
        with torch.no_grad():
            preds = model(x_numeric_val,x_category_val,x_static_val)
            loss = criterion(preds,y_val).item()
        val_loss += loss
        counter += 1
    val_loss = val_loss/counter
    print(f'Epoch {epoch} validation loss: {val_loss}')

    if val_loss<min_val_loss:
      print('saved...')
      torch.save(model,data_folder+'best.model')
      min_val_loss = val_loss

    scheduler.step()

NameError: name 'x_numeric_train_tensor' is not defined